In [2]:
import pandas as pd
import os
from pathlib import Path
from sqlalchemy import create_engine, Column, Integer, Float, Date, ForeignKey, text
from sqlalchemy.orm import sessionmaker, declarative_base, relationship
from sqlalchemy import inspect

Base = declarative_base()

# ==================== TABELAS BASE ====================

class Cliente(Base):
    __tablename__ = 'clientes'
    ID_Cliente = Column(Integer, primary_key=True)

class Produto(Base):
    __tablename__ = 'produtos'
    ID_Produto = Column(Integer, primary_key=True)

class Canal(Base):
    __tablename__ = 'canais'
    ID_Canal = Column(Integer, primary_key=True)

# ==================== NOVAS TABELAS ====================

class Venda(Base):
    __tablename__ = 'vendas'
    
    ID_Pedido = Column(Integer, primary_key=True)
    Data_Venda = Column(Date)
    Data_Entrega = Column(Date)
    ID_Cliente = Column(Integer, ForeignKey('clientes.ID_Cliente'))
    ID_Canal = Column(Integer, ForeignKey('canais.ID_Canal'))
    
    cliente = relationship("Cliente")
    canal = relationship("Canal")
    itens = relationship("ItemVenda", back_populates="venda")


class ItemVenda(Base):
    __tablename__ = 'itens_vendas'
    
    ID_Pedido = Column(Integer, ForeignKey('vendas.ID_Pedido'), primary_key=True)
    ID_Produto = Column(Integer, ForeignKey('produtos.ID_Produto'), primary_key=True)
    
    Qtde = Column(Integer)
    Valor_Unitario = Column(Float)
    Valor_Total = Column(Float)
    
    produto = relationship("Produto")
    venda = relationship("Venda", back_populates="itens")

diretorio_atual = Path(os.getcwd())

ARQUIVO_VENDAS = diretorio_atual.parent / 'planilhas' / 'vendas_2018_2021.xlsx'            
BANCO_DADOS = diretorio_atual.parent / 'data' / 'DBVendas.db'

# ==================== IMPORTAÇÃO ====================

def importar_vendas_orm(arquivo_excel=ARQUIVO_VENDAS, banco_dados=BANCO_DADOS):
    
    print("="*60)
    print("🚀 IMPORTANDO VENDAS (MODELO NORMALIZADO)")
    print("="*60)
    
    engine = create_engine(f'sqlite:///{banco_dados}', echo=False)
    
    with engine.connect() as conn:
        conn.execute(text("PRAGMA foreign_keys = ON"))
        conn.commit()
    
    inspector = inspect(engine)
    tabelas_existentes = inspector.get_table_names()
    
    print("\n🔍 Verificando tabelas necessárias...")
    for tabela in ['clientes', 'produtos', 'canais']:
        if tabela not in tabelas_existentes:
            print(f"❌ Tabela '{tabela}' não encontrada!")
            return False
        print(f"✅ {tabela}")
    
    print("\n📦 Criando tabelas vendas e itens_vendas...")
    
#    ItemVenda.__table__.drop(engine, checkfirst=True)
#    Venda.__table__.drop(engine, checkfirst=True)
    
    Base.metadata.create_all(engine)
    
    print("✅ Tabelas criadas")
    
    print(f"\n📥 Lendo Excel: {arquivo_excel}")
    df = pd.read_excel(arquivo_excel)
    print(f"📊 {len(df)} registros")
    
    # ==================== TRATAMENTO ====================
    
    df['Data_Venda'] = pd.to_datetime(df['Data_Venda'], errors='coerce').dt.date
    df['Data_Entrega'] = pd.to_datetime(df['Data_Entrega'], errors='coerce').dt.date
    df['Valor_Unitario'] = (df['Valor Total'] / df['Qtde']).round(2)
    df = df.rename(columns={'Valor Total': 'Valor_Total'})
    
    for col in ['ID_Produto', 'ID_Canal', 'ID_Cliente', 'ID_Pedido', 'Qtde']:
        df[col] = df[col].astype(int)
    
    Session = sessionmaker(bind=engine)
    session = Session()
    
    print("\n🔍 Carregando IDs válidos...")
    
    clientes_validos = set(c[0] for c in session.query(Cliente.ID_Cliente))
    produtos_validos = set(p[0] for p in session.query(Produto.ID_Produto))
    canais_validos = set(c[0] for c in session.query(Canal.ID_Canal))
    
    # ==================== INSERÇÃO ====================
    
    print("\n💾 Inserindo dados...")
    
    pedidos_criados = set()
    registros = 0
    
    for _, row in df.iterrows():
        
        if row['ID_Cliente'] not in clientes_validos:
            continue
        if row['ID_Produto'] not in produtos_validos:
            continue
        if row['ID_Canal'] not in canais_validos:
            continue
        
        # ==================== VENDAS (HEADER) ====================
        if row['ID_Pedido'] not in pedidos_criados:
            venda = Venda(
                ID_Pedido=row['ID_Pedido'],
                Data_Venda=row['Data_Venda'],
                Data_Entrega=row['Data_Entrega'],
                ID_Cliente=row['ID_Cliente'],
                ID_Canal=row['ID_Canal']
            )
            session.add(venda)
            pedidos_criados.add(row['ID_Pedido'])
        
        # ==================== ITENS ====================
        item = ItemVenda(
            ID_Pedido=row['ID_Pedido'],
            ID_Produto=row['ID_Produto'],
            Qtde=row['Qtde'],
            Valor_Unitario=row['Valor_Unitario'],
            Valor_Total=row['Valor_Total']
        )
        
        session.add(item)
        registros += 1
        
        if registros % 1000 == 0:
            session.commit()
            print(f"📝 {registros} registros...")
    
    session.commit()
    
    print("\n✅ FINALIZADO")
    print(f"📦 Pedidos: {len(pedidos_criados)}")
    print(f"📦 Itens: {registros}")
    
    session.close()
    engine.dispose()
    return True


# ==================== EXECUÇÃO ====================

if __name__ == "__main__":
    importar_vendas_orm()

🚀 IMPORTANDO VENDAS (MODELO NORMALIZADO)

🔍 Verificando tabelas necessárias...
✅ clientes
✅ produtos
✅ canais

📦 Criando tabelas vendas e itens_vendas...
✅ Tabelas criadas

📥 Lendo Excel: vendas_2018_2021.xlsx
📊 33545 registros

🔍 Carregando IDs válidos...

💾 Inserindo dados...
📝 1000 registros...
📝 2000 registros...
📝 3000 registros...
📝 4000 registros...
📝 5000 registros...
📝 6000 registros...
📝 7000 registros...
📝 8000 registros...
📝 9000 registros...
📝 10000 registros...
📝 11000 registros...
📝 12000 registros...
📝 13000 registros...
📝 14000 registros...
📝 15000 registros...
📝 16000 registros...
📝 17000 registros...
📝 18000 registros...
📝 19000 registros...
📝 20000 registros...
📝 21000 registros...
📝 22000 registros...
📝 23000 registros...
📝 24000 registros...
📝 25000 registros...
📝 26000 registros...
📝 27000 registros...
📝 28000 registros...
📝 29000 registros...
📝 30000 registros...
📝 31000 registros...
📝 32000 registros...
📝 33000 registros...

✅ FINALIZADO
📦 Pedidos: 33526
📦 Iten